In [0]:
!pip install openpyxl

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.ml.feature import Bucketizer
from pyspark.storagelevel import StorageLevel
import math

# ============================================================
# Load Excel
# ============================================================

excel_path = (
    "actual_vs_pred_sorted_by_actual_and_pred_desc_P3_P12_new_fts.xlsx"   # <-- change file name
)

pdf = pd.read_excel(excel_path)

print("Excel shape:", pdf.shape)

# Convert to Spark
actual_pred_df = spark.createDataFrame(pdf)

print("Spark shape:")
print((actual_pred_df.count(), len(actual_pred_df.columns)))

display(actual_pred_df.limit(5))

# ============================================================
# Config
# ============================================================

actual_col = "actual_TOTAL_NBRX_C4"
pred_col = "predicted_TOTAL_NBRX_C4"

source_df = actual_pred_df

# ============================================================
# Keep only needed columns and cache
# ============================================================

base_df = (
    source_df
    .select(
        F.col(actual_col).cast("double").alias(actual_col),
        F.col(pred_col).cast("double").alias(pred_col)
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = base_df.count()

df_nonzero = (
    base_df
    .filter(F.col(actual_col) > 0)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = df_nonzero.count()

In [0]:
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

# ============================================================
# Config
# ============================================================

provider_col = "PROVIDER_ID"
actual_col = "actual_TOTAL_NBRX_C4"
pred_col = "predicted_TOTAL_NBRX_C4"

# Source dataframe
source_df = actual_pred_df

# ============================================================
# Keep only needed columns and cache
# ============================================================

cols_needed = [actual_col, pred_col]
if provider_col in source_df.columns:
    cols_needed = [provider_col] + cols_needed

base_df = (
    source_df
    .select(
        *[
            F.col(c).cast("double").alias(c) if c in [actual_col, pred_col]
            else F.col(c).alias(c)
            for c in cols_needed
        ]
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = base_df.count()

# ============================================================
# Create 2 buckets only
#   Bucket 1: actual sales = 0 or 1
#   Bucket 2: actual sales > 50
# Rows with actual sales 2 to 50 are excluded
# ============================================================

bucket_1_df = (
    base_df
    .filter(F.col(actual_col).isin([0, 1, 2]))
    .withColumn("bucket_id", F.lit(1))
    .withColumn("bucket_label", F.lit("actual = 0 or 1 or 2"))
)

bucket_2_df = (
    base_df
    .filter(F.col(actual_col) > 50)
    .withColumn("bucket_id", F.lit(2))
    .withColumn("bucket_label", F.lit("actual > 50"))
)

bucketed_df = bucket_1_df.unionByName(bucket_2_df)

# ============================================================
# Summary
# ============================================================

agg_exprs = [
    F.count("*").alias("row_count"),
    F.sum(actual_col).alias("sum_actual_sales"),
    F.sum(pred_col).alias("sum_predicted_sales"),
    F.min(actual_col).alias("min_actual_sales"),
    F.max(actual_col).alias("max_actual_sales"),
    F.avg(actual_col).alias("avg_actual_sales"),
    F.avg(pred_col).alias("avg_predicted_sales")
]

if provider_col in bucketed_df.columns:
    agg_exprs.insert(
        1,
        F.countDistinct(provider_col).alias("distinct_provider_count")
    )

summary_df = (
    bucketed_df
    .groupBy("bucket_id", "bucket_label")
    .agg(*agg_exprs)
    .orderBy("bucket_id")
)

# Reorder columns nicely
if provider_col in bucketed_df.columns:
    summary_df = summary_df.select(
        "bucket_id",
        "bucket_label",
        "row_count",
        "distinct_provider_count",
        "sum_actual_sales",
        "sum_predicted_sales",
        "min_actual_sales",
        "max_actual_sales",
        "avg_actual_sales",
        "avg_predicted_sales"
    )
else:
    summary_df = summary_df.select(
        "bucket_id",
        "bucket_label",
        "row_count",
        "sum_actual_sales",
        "sum_predicted_sales",
        "min_actual_sales",
        "max_actual_sales",
        "avg_actual_sales",
        "avg_predicted_sales"
    )

display(summary_df)

In [0]:
# from pyspark.sql import functions as F
# from pyspark.sql import Window
# from pyspark.storagelevel import StorageLevel

# # ============================================================
# # Config
# # ============================================================

# actual_col = "actual_TOTAL_NBRX_C4"
# pred_col = "predicted_TOTAL_NBRX_C4"

# # Your source Spark dataframe
# # It should already exist and contain actual + predicted columns
# source_df = actual_pred_df

# # ============================================================
# # Keep only needed columns and cache
# # ============================================================

# base_df = (
#     source_df
#     .select(
#         F.col(actual_col).cast("double").alias(actual_col),
#         F.col(pred_col).cast("double").alias(pred_col)
#     )
#     .persist(StorageLevel.MEMORY_AND_DISK)
# )

# _ = base_df.count()

# df_nonzero = (
#     base_df
#     .filter(F.col(actual_col) > 0)
#     .persist(StorageLevel.MEMORY_AND_DISK)
# )

# _ = df_nonzero.count()

# # ============================================================
# # Helper: bucket summary using ntile
# # ============================================================

# def summarize_ntile_buckets(
#     df,
#     analysis_name,
#     n_buckets,
#     include_zero_bucket=False
# ):
#     """
#     If include_zero_bucket=False:
#         actual > 0 rows are split into exactly n_buckets using ntile(n_buckets)

#     If include_zero_bucket=True:
#         bucket 1 = actual == 0
#         remaining actual > 0 rows are split into exactly (n_buckets - 1) buckets
#     """

#     if include_zero_bucket:
#         # Zero bucket
#         zero_df = (
#             df.filter(F.col(actual_col) == 0)
#             .withColumn("bucket_id", F.lit(1))
#         )

#         # Non-zero rows
#         nonzero_df = df.filter(F.col(actual_col) > 0)

#         # Split non-zero rows into remaining buckets
#         remaining_buckets = n_buckets - 1

#         w = Window.orderBy(
#             F.col(actual_col).asc(),
#             F.col(pred_col).asc()
#         )

#         nonzero_bucketed_df = (
#             nonzero_df
#             .withColumn("bucket_id", F.ntile(remaining_buckets).over(w) + 1)
#         )

#         bucketed_df = zero_df.unionByName(nonzero_bucketed_df)

#     else:
#         # Split all rows in df into exactly n_buckets
#         w = Window.orderBy(
#             F.col(actual_col).asc(),
#             F.col(pred_col).asc()
#         )

#         bucketed_df = df.withColumn(
#             "bucket_id",
#             F.ntile(n_buckets).over(w)
#         )

#     # Summary per bucket
#     summary_df = (
#         bucketed_df
#         .groupBy("bucket_id")
#         .agg(
#             F.count("*").alias("hcp_count"),
#             F.sum(actual_col).alias("sum_actual_sales"),
#             F.sum(pred_col).alias("sum_predicted_sales"),
#             F.min(actual_col).alias("min_actual_sales"),
#             F.max(actual_col).alias("max_actual_sales"),
#             F.avg(actual_col).alias("avg_actual_sales"),
#             F.avg(pred_col).alias("avg_predicted_sales")
#         )
#         .orderBy("bucket_id")
#     )

#     # Add readable bucket labels
#     if include_zero_bucket:
#         label_rows = [(1, "actual = 0")]

#         for i in range(2, n_buckets + 1):
#             label_rows.append((i, f"non-zero bucket {i - 1}"))

#     else:
#         label_rows = [(i, f"bucket {i}") for i in range(1, n_buckets + 1)]

#     label_df = spark.createDataFrame(label_rows, ["bucket_id", "bucket_label"])

#     final_df = (
#         summary_df
#         .join(label_df, on="bucket_id", how="left")
#         .select(
#             F.lit(analysis_name).alias("analysis_name"),
#             "bucket_id",
#             "bucket_label",
#             "hcp_count",
#             "sum_actual_sales",
#             "sum_predicted_sales",
#             "min_actual_sales",
#             "max_actual_sales",
#             "avg_actual_sales",
#             "avg_predicted_sales"
#         )
#         .orderBy("bucket_id")
#     )

#     return final_df

# # ============================================================
# # 3-bucket analysis
# #   - all rows: bucket 1 = 0, buckets 2-3 = non-zero split into 2 buckets
# #   - nonzero only: 3 exact buckets
# # ============================================================

# summary_all_3 = summarize_ntile_buckets(
#     df=base_df,
#     analysis_name="all_rows_3_buckets",
#     n_buckets=3,
#     include_zero_bucket=True
# )

# summary_nonzero_3 = summarize_ntile_buckets(
#     df=df_nonzero,
#     analysis_name="actual_gt_0_3_buckets",
#     n_buckets=3,
#     include_zero_bucket=False
# )

# # ============================================================
# # 10-bucket analysis
# #   - all rows: bucket 1 = 0, buckets 2-10 = non-zero split into 9 buckets
# #   - nonzero only: 10 exact buckets
# # ============================================================

# summary_all_10 = summarize_ntile_buckets(
#     df=base_df,
#     analysis_name="all_rows_10_buckets",
#     n_buckets=10,
#     include_zero_bucket=True
# )

# summary_nonzero_10 = summarize_ntile_buckets(
#     df=df_nonzero,
#     analysis_name="actual_gt_0_10_buckets",
#     n_buckets=10,
#     include_zero_bucket=False
# )

# # ============================================================
# # Show results
# # ============================================================

# print("=== ALL ROWS | 3 BUCKETS ===")
# display(summary_all_3)

# print("=== ACTUAL > 0 | 3 BUCKETS ===")
# display(summary_nonzero_3)

# print("=== ALL ROWS | 10 BUCKETS ===")
# display(summary_all_10)

# print("=== ACTUAL > 0 | 10 BUCKETS ===")
# display(summary_nonzero_10)

# # ============================================================
# # Optional combined output
# # ============================================================

# all_summaries = (
#     summary_all_3
#     .unionByName(summary_nonzero_3)
#     .unionByName(summary_all_10)
#     .unionByName(summary_nonzero_10)
# )

# print("=== COMBINED OUTPUT ===")
# display(all_summaries)